In [28]:
import pandas as pd
#%pip install sweetviz
import sweetviz as sv
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression



**1. (3 points)** Using pandas, load the dataset and display the first 5 rows.

In [29]:
insurance = pd.read_csv('/Users/amritdhillon/Desktop/Advanced ML/insurance.csv')
insurance.head(5)


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


**2. (3 points)** Using SweetViz, do a quick EDA of the data set. In a markdown cell describe the main trends and patterns you see in the data.

In [30]:
report = sv.analyze(insurance)
report.show_html('insurance_report.html')

                                             |          | [  0%]   00:00 -> (? left)

Report insurance_report.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


Trends + Patterns in Data: 

- **Smoker status** has a strong impact on insurance charges as smokers tend to have significantly higher charges compared to non-smokers
- **Age** is positively correlated with charges, with older individuals generally having higher medical costs
- **BMI** also shows a positive relationship with charges, especially for individuals with higher BMI
- **Number of children** appears to have a relatively small effect on charges compared to other variables
- The dataset appears to have no major missing values and is fairly well-balanced across things like sex and region

Overall, smoker status, age, and BMI are the most important variables affecting insurance charges

**3. (4 points)** Define the input variables (X) and target variable (y). Split the data into training (80%) and testing (20%). 

In [31]:
insurance_encoded = pd.get_dummies(insurance, drop_first = True).astype(int)


X = insurance_encoded.drop('charges', axis = 1)
y = insurance_encoded['charges']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

**4. (4 points)** Scale the input variables.

In [32]:
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

**5. (5 points)** Build a baseline linear regressor and random forest regressor. Report both R2 and MSE for both models. Add a markdown cell discussing the performance of each of the models on the test data. 

In [33]:
baseline_lr = LinearRegression()
baseline_lr.fit(X_train_scaled, y_train)
y_pred_lr = baseline_lr.predict(X_test_scaled)
mse_lr = mean_squared_error(y_test, y_pred_lr)
r2_lr = r2_score(y_test, y_pred_lr)

baseline_rf = RandomForestRegressor()
baseline_rf.fit(X_train_scaled, y_train)
y_pred_rf = baseline_rf.predict(X_test_scaled)
mse_rf = mean_squared_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)

print(f'Baseline LR MSE {mse_lr:.2f}, R2 : {r2_lr:.2f}')
print(f'Baseline RF MSE {mse_rf:.2f}, R2 : {r2_rf:.2f}')

Baseline LR MSE 33566439.74, R2 : 0.78
Baseline RF MSE 21751010.88, R2 : 0.86


Model Performance on Test Data:

- The **Linear Regression model** had an MSE of 33,566,439.74 and an R² of 0.78. This indicates that the model explains about 78% of the variance in insurance charges. While this is a strong baseline, Linear Regression assumes a linear relationship between the features and the target, limiting its ability to capture complex patterns in the data.

- The **Random Forest model** had a lower MSE of 22,094,641.93 and a higher R² of 0.86. This means it explains approximately 86% of the variance while having considerably smaller mean squared error, significantly improving upon the Linear Regression model.

Overall, the Random Forest model outperforms Linear Regression, as indicated by its higher R² and lower MSE. This improvement is likely due to its ability to capture non-linear relationships and interactions between variables such as smoker status, age, and BMI.

In [34]:
#check against cross validated, not needed for any of questions I dont think, but we did in class
cv_lr = cross_val_score(baseline_lr, X_train_scaled, y_train, cv = 5, scoring = 'r2')
cv_rf = cross_val_score(baseline_rf, X_train, y_train, cv = 5, scoring = 'r2')
print(f'Baseline LR CV R2 Scores: {cv_lr}')
print(f'Baseline RF CV R2 Scores: {cv_rf}')

Baseline LR CV R2 Scores: [0.71508701 0.8018118  0.72333011 0.65748491 0.76763199]
Baseline RF CV R2 Scores: [0.81728461 0.89793348 0.78914031 0.78268335 0.82940057]


**6. (5 points)** Use GridSearchCV to tune n_estimators, max_depth, min_samples_split and max_features. What is the best set of hyperparameters from your search?

In [35]:
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10],
    'max_features': ['sqrt', 'log2']
}


grid_search = GridSearchCV(
    estimator=baseline_rf,
    param_grid=param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("Best Random Forest Hyperparameters:", grid_search.best_params_)
print(f'Best Random Forest Cross-Validated R2 Score: {grid_search.best_score_:.2f}')

Best Random Forest Hyperparameters: {'max_depth': 10, 'max_features': 'log2', 'min_samples_split': 5, 'n_estimators': 100}
Best Random Forest Cross-Validated R2 Score: 0.84


**7. (5 points)** Examine the top 20 models (based on rank) from your GridSearchCV results. Add a markdown cell and discuss which set of hyperparameters you would choose and why. 

In [36]:
# examine top 20 ranked parameter combinations from grid search 
print("\nTop 20 GridSearchCV results:")
top20_eval = (pd.DataFrame(grid_search.cv_results_).sort_values('mean_test_score', ascending=False).head(20)[['param_n_estimators', 'param_max_depth', 'param_min_samples_split', 
'param_max_features', 'mean_test_score']].rename(columns=lambda x: x.replace('param_', '')))

print(top20_eval)


Top 20 GridSearchCV results:
    n_estimators max_depth  min_samples_split max_features  mean_test_score
20           100        10                  5         log2         0.841297
8            100      None                  5         log2         0.840658
21           200        10                  5         log2         0.840578
9            200      None                  5         log2         0.839838
34           100        20                 10         log2         0.839783
35           200        20                 10         log2         0.839160
22           100        10                 10         log2         0.838802
32           100        20                  5         log2         0.838534
33           200        20                  5         log2         0.838527
19           200        10                  2         log2         0.838370
23           200        10                 10         log2         0.837958
10           100      None                 10         log2

Chosen Set of Hyperparameters + Explanation: 

- Although the top-ranked model achieved the highest R² score (**0.8409**), the improvement over other models is extremely small (only about 0.0005-.0006)

For example, the simpler model with:
- n_estimators = 100  
- max_depth = None  
- min_samples_split = 5  
- max_features = log2  

has a nearly identical R² score of **0.8403**.

Given the negligible difference in performance, the simpler model is preferred. It requires fewer trees, reducing computational cost, while maintaining nearly the same predictive accuracy. This shows that small improvements in cross-validation score do not necessarily justify increased model complexity.

**8. (3 points--if time)** Compare the training versus test performance using a Random Forest model tuned with the best parameters from GridSearch CV. Is your model overfitting and how do you know?

In [37]:
# Final model using chosen hyperparameters
final_rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=None,
    min_samples_split=5,
    max_features='log2',
    random_state=42
)

final_rf.fit(X_train, y_train)

#Training performance
y_train_pred = final_rf.predict(X_train)
train_r2 = r2_score(y_train, y_train_pred)
train_mse = mean_squared_error(y_train, y_train_pred)
print(f'Training R2: {train_r2:.2f}')
print(f'Training MSE: {train_mse:.2f}')

#Test performance
y_pred_final = final_rf.predict(X_test)
mse_final = mean_squared_error(y_test, y_pred_final)
r2_final = r2_score(y_test, y_pred_final)
print(f'Final RF MSE: {mse_final:.2f}')
print(f'Final RF R2: {r2_final:.2f}')

Training R2: 0.94
Training MSE: 8974436.31
Final RF MSE: 20673918.65
Final RF R2: 0.87


In [38]:
#compare back to the original for reference
print(f'Baseline RF MSE: {mse_rf:.2f}')
print(f'Baseline RF R2: {r2_rf:.2f}')


Baseline RF MSE: 21751010.88
Baseline RF R2: 0.86


Is model overfitting and how do you know: 

- The final Random Forest model shows some difference between training and test performance, but it does not appear to be severely overfitting.

- The model achieves an R² of 0.94 on the training data and 0.87 on the test data. While the training performance is higher, which is expected, the gap is moderate and indicates that the model is still generalizing well to unseen data.

- Additionally, the test performance improved compared to the baseline model (R² increased from 0.86 to 0.87 and MSE decreased), suggesting that hyperparameter tuning helped improve prediction rather than causing overfitting.

- Overall, the model has a small amount of overfitting, but this is typical for models like Random Forests, and the difference is not large enough to be worried about.